In [1]:
import pandas as pd
import numpy as np
import joblib
import faiss

from sentence_transformers import SentenceTransformer

1.14.3


In [2]:
counsel_df = pd.read_csv("../datasets/Counseling Conversations.csv")

print(counsel_df.shape)
print(counsel_df.head())

(3512, 2)
                                             Context  \
0  I'm going through some things with my feelings...   
1  I'm going through some things with my feelings...   
2  I'm going through some things with my feelings...   
3  I'm going through some things with my feelings...   
4  I'm going through some things with my feelings...   

                                            Response  
0  If everyone thinks you're worthless, then mayb...  
1  Hello, and thank you for your question and see...  
2  First thing I'd suggest is getting the sleep y...  
3  Therapy is essential for those that are feelin...  
4  I first want to let you know that you are not ...  


In [3]:
counsel_df = counsel_df.dropna()
counsel_df = counsel_df.reset_index(drop=True)

print(counsel_df.shape)

(3508, 2)


In [4]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L12-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
embeddings = model.encode(
    counsel_df["Context"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/110 [00:00<?, ?it/s]

(3508, 384)


In [6]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Total vectors:", index.ntotal)

Total vectors: 3508


In [7]:
faiss.write_index(index, "../models/counseling_faiss.index")

In [8]:
joblib.dump(counsel_df, "../models/counseling_dataset.pkl")

['../models/counseling_dataset.pkl']

In [9]:
query = "I feel lonely and depressed."
query_embedding = model.encode(
    [query],
    convert_to_numpy=True
)
distances, indices = index.search(query_embedding, k=3)

for i in indices[0]:
    print("=" * 80)
    print("Context:")
    print(counsel_df.iloc[i]["Context"])
    print("\nResponse:")
    print(counsel_df.iloc[i]["Response"])

Context:
I live a normal life. I have tons of friends and family, but I feel lonely.

Response:
This may be happening because you and the others are not connected to each other on a level which reaches your emotions.Loneliness may show the absence of feeling a variety of emotions when you are among others.How many friends you have doesn't affect whether you and someone else feel emotionally engaged with one another.Consider if you feel like concentrating your friendship on more intensively sharing your feelings with a few of your friends.This may lead to fewer friends who are also more meaningful to you and your feeling a decrease of loneliness.
Context:
I live a normal life. I have tons of friends and family, but I feel lonely.

Response:
This may be happening because you and the others are not connected to each other on a level which reaches your emotions.Loneliness may show the absence of feeling a variety of emotions when you are among others.How many friends you have doesn't affec

In [10]:
import joblib
import faiss
from sentence_transformers import SentenceTransformer

# Load Notebook 3 models
tfidf = joblib.load("../models/tfidf_vectorizer.pkl")
emotion_model = joblib.load("../models/main_emotion_model_tfidf.pkl")
main_encoder = joblib.load("../models/main_emotion_encoder.pkl")

# Load Notebook 4
index = faiss.read_index("../models/counseling_faiss.index")
counsel_df = joblib.load("../models/counseling_dataset.pkl")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L12-v2"
)

def predict_and_retrieve(user_text, top_k=3):

    # ---------- Emotion Prediction ----------
    x = tfidf.transform([user_text])

    pred = emotion_model.predict(x)[0]

    main_emotion = main_encoder.inverse_transform([pred])[0]

    # ---------- RAG Retrieval ----------
    query_embedding = embedding_model.encode(
        [user_text],
        convert_to_numpy=True
    )

    distances, indices = index.search(query_embedding, top_k)

    retrieved = []

    for idx in indices[0]:
        retrieved.append({
            "context": counsel_df.iloc[idx]["Context"],
            "response": counsel_df.iloc[idx]["Response"]
        })

    return {
        "main_emotion": main_emotion,
        "retrieved": retrieved
    }

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [11]:
result = predict_and_retrieve(
    "I feel lonely and hopeless."
)

print("Predicted Emotion:", result["main_emotion"])

for i, item in enumerate(result["retrieved"], 1):
    print("\n", "="*80)
    print(f"Retrieved {i}")
    print(item["context"])
    print()
    print(item["response"])

Predicted Emotion: Sad

Retrieved 1
I live a normal life. I have tons of friends and family, but I feel lonely.

This may be happening because you and the others are not connected to each other on a level which reaches your emotions.Loneliness may show the absence of feeling a variety of emotions when you are among others.How many friends you have doesn't affect whether you and someone else feel emotionally engaged with one another.Consider if you feel like concentrating your friendship on more intensively sharing your feelings with a few of your friends.This may lead to fewer friends who are also more meaningful to you and your feeling a decrease of loneliness.

Retrieved 2
I live a normal life. I have tons of friends and family, but I feel lonely.

This may be happening because you and the others are not connected to each other on a level which reaches your emotions.Loneliness may show the absence of feeling a variety of emotions when you are among others.How many friends you have do